# Mixar — build Linux (x86_64) no Kaggle

Compila a versão **mais recente** do [Mixar](https://github.com/Mixar-AI/mixar-app) a partir do código-fonte
público, aplica os patches de [`aimcrazyxx/mixar-app3`](https://github.com/aimcrazyxx/mixar-app3)
(`patches/series`) e empacota o app em `/kaggle/working/mixar-<versão>-linux-x64.tar.xz`.

* Requer **Internet ligada** (Settings → Internet). CPU basta; GPU não acelera a compilação.
* Leva de 3 a 4 horas em 4 vCPUs (limite do Kaggle: 12 h).
* Saída: o `.tar.xz`, o `.sha256`, `BUILD_INFO.txt` e `logs/` (log completo do build, teste de fumaça).

In [ ]:
# Parâmetros
MIXAR_APP3_REPO = "https://github.com/aimcrazyxx/mixar-app3.git"
MIXAR_APP3_REF = "main"   # branch/tag deste repositório com o script de build e os patches
MIXAR_REF = "latest-release"  # Mixar oficial: "latest-release" (release mais nova), "main" (código mais novo) ou uma tag, ex. "v4.3.4"
MIXAR_ENV = "Prod"        # Prod | Dev | UAT

In [ ]:
import os
import subprocess

BUILD_REPO = "/tmp/mixar-app3"
subprocess.run(["rm", "-rf", BUILD_REPO], check=True)
clone = ["git", "clone", "--depth", "1", MIXAR_APP3_REPO, BUILD_REPO]
if subprocess.run(clone[:4] + ["--branch", MIXAR_APP3_REF] + clone[4:]).returncode != 0:
    # A merged and deleted branch should not break the notebook: fall back to main.
    print(f"Branch {MIXAR_APP3_REF!r} indisponível; usando a branch padrão")
    subprocess.run(["rm", "-rf", BUILD_REPO], check=True)
    subprocess.run(clone, check=True)
print(subprocess.run(["git", "-C", BUILD_REPO, "log", "-1", "--format=%h %s"],
                     check=True, capture_output=True, text=True).stdout)

In [ ]:
env = dict(os.environ, MIXAR_REF=MIXAR_REF, MIXAR_ENV=MIXAR_ENV)
proc = subprocess.Popen(["bash", f"{BUILD_REPO}/kaggle/build_mixar_linux.sh"], env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
if proc.wait() != 0:
    raise RuntimeError(f"Build falhou (código {proc.returncode}); veja /kaggle/working/logs/")

In [ ]:
print(open("/kaggle/working/BUILD_INFO.txt").read())
subprocess.run(["ls", "-lh", "/kaggle/working", "/kaggle/working/logs"])